# QuantStream — phòng thực hành fine-tuning Chronos-2 + LoRA

**Chạy riêng sau notebook 04**, dùng cùng Dataset input mới, Internet On và GPU còn quota. Notebook này tải pretrained model, học adapter và so sánh với zero-shot trên cùng 512 cửa sổ development. Không chạy đồng thời với notebook 04.

Đây là fine-tuning thật: cập nhật adapter trọng số từ mô hình đã pretrained. TCN/GRU ở notebook 04 được train từ đầu. Pretraining overlap chưa xác minh nên kết quả Chronos chỉ phục vụ nghiên cứu/học tập, không chứng minh alpha độc lập.

Ba cấu hình được đặt trước: (100 steps, LR 1e-5), (200 steps, LR 1e-5), (200 steps, LR 3e-6). So A/B để học tác động số bước, B/C để học tác động learning rate. Giữ seed, context, batch size, base revision và cửa sổ chấm điểm cố định.

In [ ]:
import subprocess,sys,os
os.environ['HF_HUB_DISABLE_XET']='1'
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'chronos-forecasting==2.3.2','peft>=0.18.1,<1','accelerate>=1.1,<2',
    'datasets>=3,<5','pyarrow>=17,<27','scikit-learn>=1.5,<2'])

In [ ]:
from pathlib import Path
import sys, zipfile, shutil, json, hashlib
WORK=Path('/kaggle/working/quantstream_research') if Path('/kaggle/working').exists() else Path.cwd()/'notebook_work'
WORK.mkdir(parents=True,exist_ok=True)
INPUT=Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd().parent
archives=list(INPUT.rglob('QuantStream_Kaggle_Input.zip'))
if len(archives)>1:
    raise ValueError('Chỉ Add Input một phiên bản QuantStream dataset; bỏ các phiên bản cũ.')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for name in z.namelist():
            p=Path(name)
            if p.is_absolute() or '..' in p.parts: raise ValueError('Invalid ZIP path')
        z.extractall(WORK)
else:
    manifests=[p for p in INPUT.rglob('manifest.json') if (p.parent/'development.parquet').exists()]
    if len(manifests)!=1: raise FileNotFoundError('Add Input: QuantStream_Kaggle_Input.zip bản mới nhất.')
    source=manifests[0].parent
    shutil.copytree(source,WORK/'snapshot',dirs_exist_ok=True)
    shutil.copytree(source.parent/'code',WORK/'code',dirs_exist_ok=True)
SNAPSHOT=WORK/'snapshot'
manifest=json.loads((SNAPSHOT/'manifest.json').read_text())
if manifest.get('contains_final_holdout') is not False: raise ValueError('Development-only input required')
if not manifest.get('code_files') or not (WORK/'code/quantstream/search.py').exists():
    raise ValueError('Input cũ: tải lại ZIP mới dành cho notebook 04/05.')
for name,expected in manifest['code_files'].items():
    if hashlib.sha256((WORK/'code'/name).read_bytes()).hexdigest()!=expected:
        raise ValueError('Source checksum mismatch: '+name)
sys.path.insert(0,str(WORK/'code'))
print('Source:',manifest['source_control'])
print('Development only. Final holdout included:',manifest['contains_final_holdout'])

In [ ]:
import torch,gc
from quantstream.search import load_snapshot
from quantstream.finetune import chronos_experiment
from IPython.display import display,FileLink
load_snapshot(SNAPSHOT)  # Verify data hashes and reject final-holdout contamination.
if not torch.cuda.is_available():
    raise RuntimeError('Lab này cần GPU để giữ ngân sách thời gian. Chọn GPU khả dụng trước khi chạy.')
print(torch.__version__,torch.cuda.get_device_name(0))
BASE_REVISION='29ec3766d36d6f73f0696f85560a422f50e8498c'
CONFIGS=[{'id':'A_steps100_lr1e5','steps':100,'learning_rate':1e-5},
         {'id':'B_steps200_lr1e5','steps':200,'learning_rate':1e-5},
         {'id':'C_steps200_lr3e6','steps':200,'learning_rate':3e-6}]
OUTPUT=WORK/'run-003-lora-lab'
OUTPUT.mkdir(exist_ok=True)
plan={'base_revision':BASE_REVISION,'configurations':CONFIGS,'seed':42,
      'context_length':128,'batch_size':8,'max_windows':512,
      'snapshot_manifest_sha256':hashlib.sha256((SNAPSHOT/'manifest.json').read_bytes()).hexdigest()}
from quantstream.common import write_json
if (OUTPUT/'plan.json').exists() and json.loads((OUTPUT/'plan.json').read_text())!=plan:
    raise ValueError('Không thay đổi protocol của thí nghiệm đã bắt đầu.')
write_json(OUTPUT/'plan.json',plan)

## Fine-tune và xuất evidence

Mỗi cấu hình bắt đầu từ **cùng pretrained checkpoint**, không nối tiếp adapter A → B → C. Nếu B kém A, chưa thể kết luận thêm bước luôn xấu; cần xem loss/seed ở vòng nghiên cứu sau. Không chỉnh cấu hình ngay sau khi thấy kết quả để rồi gọi validation là test.

Output giữ adapter, báo cáo, cùng các hàng dự báo zero-shot/LoRA. Các optimizer checkpoint lớn được bỏ khỏi ZIP chia sẻ. Nếu phiên bị ngắt, Save Version/Download output; không xóa bằng chứng trial lỗi.

In [ ]:
reports=[]
try:
    for spec in CONFIGS:
        folder=OUTPUT/spec['id']
        report_path=folder/'chronos_report.json'
        if report_path.exists():
            report=json.loads(report_path.read_text())
        else:
            report=chronos_experiment(SNAPSHOT,folder,steps=spec['steps'],
                learning_rate=spec['learning_rate'],max_windows=512,device='cuda',
                revision=BASE_REVISION,batch_size=8,context_length=128,seed=42)
        reports.append({'experiment':spec['id'],**report})
        gc.collect();torch.cuda.empty_cache()
    write_json(OUTPUT/'comparison.json',reports)
finally:
    # Package adapter/evidence only, while retaining trainer logs in Kaggle Output.
    with zipfile.ZipFile(WORK.parent/'QuantStream_LoRA_Output.zip','w',zipfile.ZIP_DEFLATED) as z:
        for path in OUTPUT.rglob('*'):
            if path.is_file() and 'trainer' not in path.relative_to(OUTPUT).parts:
                z.write(path,path.relative_to(OUTPUT))
    display(FileLink(str(WORK.parent/'QuantStream_LoRA_Output.zip')))

In [ ]:
import pandas as pd
table=pd.DataFrame(reports)
table['mae_change']=table.finetuned_mae-table.zero_shot_mae
display(table[['experiment','steps','learning_rate','zero_shot_mae','finetuned_mae',
               'mae_change','zero_shot_directional_accuracy','finetuned_directional_accuracy']])
print('mae_change < 0: sai số giảm trên các cửa sổ development này. Chưa phải lợi nhuận giao dịch.')
print('Gửi QuantStream_LoRA_Output.zip để review adapter và ghi kết quả lên Git.')